<a href="https://colab.research.google.com/github/Chathu283/Statistical-Learning-e23218/blob/main/Assignment%207c%3A%20Item%20Response%20Prediction%20and%20Click%20Through%20Rate%20Prediction_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses
## Answers

## Task 1: Visualizing the Mechanics of the 2PL Item Response Model

The Two-Parameter Logistic (2PL) Item Response Theory (IRT) model describes the probability that a user with latent ability $\theta$ correctly answers item $i$.

The response probability is

$$
P(Y_i=1 \mid \Theta=\theta)=p_i(\theta)
=\frac{1}{1+\exp[-a_i(\theta-b_i)]},
$$

where

- $\theta$ is the user's latent ability,
- $a_i>0$ is the discrimination parameter,
- $b_i$ is the difficulty parameter.

The discrimination parameter controls the steepness of the logistic curve, while the difficulty parameter controls its horizontal position.

### Interpretation of the Parameters

The probability satisfies

$$
0\le p_i(\theta)\le1.
$$

When

$$
\theta=b_i,
$$

the probability becomes

$$
p_i(\theta)=\frac12.
$$

Therefore, the difficulty parameter represents the ability level required to have a 50% probability of answering correctly.

---

### Effect of the Difficulty Parameter

Suppose the discrimination is fixed as

$$
a_i=1.5,
$$

while

$$
b_i=-1,\;0,\;1.
$$

Then

- Increasing $b_i$ shifts the curve to the **right**.
- Decreasing $b_i$ shifts the curve to the **left**.
- The overall shape of the curve remains unchanged.

Mathematically,

$$
p_i(\theta)
=
\frac{1}
{1+\exp[-a_i(\theta-b_i)]},
$$

contains the quantity

$$
(\theta-b_i),
$$

which is simply a horizontal translation.

---

### Effect of the Discrimination Parameter

Suppose

$$
b_i=0
$$

is fixed while

$$
a_i=0.5,\;2.0.
$$

Then

- Small values of $a_i$ produce a gradual S-shaped curve.
- Large values of $a_i$ produce a much steeper curve.

The derivative of the logistic function is

$$
\frac{dp_i}{d\theta}
=
a_i\,p_i(\theta)\left[1-p_i(\theta)\right].
$$

Thus the slope is directly proportional to the discrimination parameter.

Larger values of $a_i$ imply that a small increase in ability produces a larger increase in the probability of answering correctly.

In [3]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4,4,500)

def irt(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

fig = go.Figure()

# Same discrimination, different difficulties
for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=irt(theta,1.5,b),
            mode="lines",
            name=f"a=1.5, b={b}"
        )
    )

# Different discrimination
fig.add_trace(
    go.Scatter(
        x=theta,
        y=irt(theta,0.5,0),
        mode="lines",
        name="a=0.5, b=0",
        line=dict(dash='dash')
    )
)

fig.add_trace(
    go.Scatter(
        x=theta,
        y=irt(theta,2.0,0),
        mode="lines",
        name="a=2.0, b=0",
        line=dict(dash='dot')
    )
)

fig.update_layout(
    title="2PL Item Characteristic Curves",
    xaxis_title="Ability (θ)",
    yaxis_title="P(Correct)",
    template="plotly_white"
)

fig.show()

# Task 2: Sequential Likelihood Contribution

For a single observation,

$$
Y_k\in\{0,1\},
$$

the conditional probability is

$$
P(Y_k=y_k\mid\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}.
$$

Hence the likelihood contribution of one newly observed response is

$$
\boxed{
L(y_k|\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}
}
$$

where

$$
p_k(\theta)
=
\frac{1}
{1+\exp[-a_k(\theta-b_k)]}.
$$

---

Because the responses are conditionally independent,

$$
Y_1,\ldots,Y_k
$$

have joint likelihood

$$
L(\mathbf y^{(k)}|\theta)
=
\prod_{i=1}^{k}
P(Y_i=y_i|\theta).
$$

Substituting the Bernoulli probabilities,

$$
\boxed{
L(\mathbf y^{(k)}|\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left(1-p_i(\theta)\right)^{1-y_i}
}
$$

This likelihood summarizes all information collected from the first $k$ items.

# Task 3: Recursive Posterior Update

Initially,

$$
\Theta\sim N(0,1),
$$

with prior density

$$
f_\Theta^{(0)}(\theta)
=
\frac1{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

Suppose that after observing

$$
\mathbf y^{(k-1)},
$$

the posterior density is

$$
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf y^{(k-1)}).
$$

When the next response

$$
y_k
$$

arrives, Bayes' theorem gives

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf y^{(k)})
=
\frac{
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf y^{(k-1)})
}
{
\int
L(y_k|t)
f_{\Theta|Y^{(k-1)}}(t|\mathbf y^{(k-1)})
dt
}.
$$

Ignoring the normalization constant,

$$
\boxed{
f_{\Theta|Y^{(k)}}(\theta)
\propto
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}(\theta)
}
$$

This recursive equation forms the basis of online Bayesian updating.

# Task 4: Dynamic Shifting of the Posterior

Suppose the user answers correctly,

$$
y_k=1.
$$

The likelihood becomes

$$
L(y_k|\theta)
=
p_k(\theta).
$$

If the item has a large difficulty,

$$
b_k\gg0,
$$

then

$$
p_k(\theta)
=
\frac1{1+\exp[-a_k(\theta-b_k)]}
$$

is appreciable only for relatively large values of

$$
\theta.
$$

Multiplying the previous posterior by this likelihood,

$$
f_{\Theta|Y^{(k)}}(\theta)
\propto
p_k(\theta)
f_{\Theta|Y^{(k-1)}}(\theta),
$$

assigns greater weight to larger ability values.

Consequently,

- the posterior peak shifts toward larger values of $\theta$,
- the posterior mean increases,
- the MAP estimate also moves to the right.

Therefore, answering a difficult item correctly increases the estimated ability more than answering an easy item correctly.

# Task 5: Tracking Certainty and Sharpness

The discrimination parameter determines how strongly the item distinguishes users with different ability levels.

The derivative of the logistic curve is

$$
\frac{dp(\theta)}{d\theta}
=
a_k\,p(\theta)\left(1-p(\theta)\right).
$$

Hence,

- Larger $a_k$ produces a steeper likelihood.
- Smaller $a_k$ produces a flatter likelihood.

During Bayesian updating,

$$
f_{\Theta|Y^{(k)}}
\propto
L(y_k|\theta)
f_{\Theta|Y^{(k-1)}}.
$$

When

$$
a_k
$$

is very large, the likelihood is highly concentrated around the most likely ability value.

Therefore,

- posterior variance decreases,
- posterior becomes narrower,
- uncertainty decreases,
- confidence in the user's ability increases.

Conversely, when

$$
a_k
$$

is very small, the likelihood is nearly flat.

The observation contributes little new information, causing

- posterior variance to remain relatively large,
- posterior to remain broad,
- slower convergence of the ability estimate.

Hence, highly discriminating questions provide much more information about the user's latent ability than weakly discriminating questions.

# Task 6: Numerical Implementation of a Running Grid

## Numerical Approximation of the Posterior Distribution

The posterior distribution under the 2PL Item Response Theory (IRT) model does not have a closed-form analytical solution because the logistic likelihood is **not conjugate** to the Normal prior. Consequently, the posterior density must be approximated numerically.

A convenient and intuitive approach is to discretize the latent ability parameter $\theta$ over a fixed grid and perform Bayesian updating directly on this grid after each observed response.

---

## Step 1: Construct a Fixed Ability Grid

First, define a sufficiently dense grid covering the range of plausible ability values.

For example,

$$
\theta_j\in[-4,4],
\qquad j=1,\ldots,m,
$$

where

$$
m=1000
$$

grid points.

Mathematically,

$$
\Theta_{\text{grid}}
=
\{\theta_1,\theta_2,\ldots,\theta_m\}.
$$

Each grid point represents one possible value of the user's latent ability.

---

## Step 2: Initialize the Prior Distribution

The assignment specifies the prior

$$
\Theta\sim N(0,1).
$$

Therefore,

$$
f^{(0)}(\theta)
=
\frac1{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

This prior is evaluated at every grid point,

$$
f_j^{(0)}
=
f^{(0)}(\theta_j).
$$

Since numerical computation uses discrete probabilities rather than continuous densities, the prior must be normalized so that

$$
\sum_{j=1}^{m}
f_j^{(0)}
=
1
$$

Thus,

$$
f_j^{(0)}
\leftarrow
\frac{f_j^{(0)}}
{\sum_{r=1}^{m}f_r^{(0)}}.
$$

---

## Step 3: Compute the Likelihood

Suppose the current item has parameters

$$
a_k,\qquad b_k.
$$

The probability of a correct response at each grid point is

$$
p_k(\theta_j)
=
\frac1
{1+\exp[-a_k(\theta_j-b_k)]}.
$$

The likelihood depends on the observed response.

If

$$
y_k=1,
$$

then

$$
L_j
=
p_k(\theta_j).
$$

If

$$
y_k=0,
$$

then

$$
L_j
=
1-p_k(\theta_j).
$$

Combining both cases,

$$
\boxed{
L_j
=
p_k(\theta_j)^{y_k}
\left(1-p_k(\theta_j)\right)^{1-y_k}
}
$$

for every grid point.

---

## Step 4: Bayesian Update

Using Bayes' theorem,

$$
f_j^{(k)}
\propto
L_j
f_j^{(k-1)}.
$$

In practice,

$$
\tilde f_j
=
L_j
f_j^{(k-1)}
$$

is first computed.

This quantity is called the **unnormalized posterior**.

---

## Step 5: Sequential Normalization

The unnormalized posterior is not yet a probability distribution because

$$
\sum_j\tilde f_j\neq1.
$$

Therefore, the normalization constant is

$$
Z
=
\sum_{j=1}^{m}
\tilde f_j.
$$

The normalized posterior becomes

$$
\boxed{
f_j^{(k)}
=
\frac{\tilde f_j}
{Z}
}
$$

or equivalently,

$$
\boxed{
f_j^{(k)}
=
\frac{
L_j
f_j^{(k-1)}
}{
\sum_{r=1}^{m}
L_r
f_r^{(k-1)}
}
}
$$

This guarantees

$$
\sum_{j=1}^{m}
f_j^{(k)}
=
1.
$$

The normalized posterior then becomes the prior for the next item.

---

## Step 6: Posterior Mean

The Bayesian estimate of ability is the posterior expectation,

$$
\boxed{
\hat\theta_{\mathrm{Bayes}}
=
\sum_{j=1}^{m}
\theta_j
f_j^{(k)}
}
$$

which represents the center of mass of the posterior distribution.

---

## Step 7: Maximum A Posteriori (MAP)

The MAP estimate is simply the grid point corresponding to the highest posterior probability,

$$
\boxed{
\hat\theta_{\mathrm{MAP}}
=
\underset{\theta_j}{\operatorname{argmax}}
\;
f_j^{(k)}
}
$$

Unlike the posterior mean, the MAP estimate identifies the single most probable ability level.

---

## Algorithm Summary

The numerical Bayesian updating algorithm proceeds as follows:

1. Construct a fixed grid of ability values.
2. Evaluate the Normal prior on the grid.
3. Normalize the prior.
4. For each newly answered item:
   - Compute the response probability using the 2PL model.
   - Compute the likelihood.
   - Multiply the previous posterior by the likelihood.
   - Normalize the resulting posterior.
   - Compute the Posterior Mean.
   - Compute the MAP estimate.
5. Repeat until all items have been processed.

This grid-based Bayesian algorithm provides a computationally efficient approximation of the posterior distribution without requiring analytical integration or Markov Chain Monte Carlo (MCMC) sampling.

In [4]:
import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go

# Ability grid
theta = np.linspace(-4, 4, 1000)

# Initial prior N(0,1)
prior = norm.pdf(theta, 0, 1)
prior /= prior.sum()

# Example item
a = 1.5
b = 0.5

# Assume the user answered correctly
y = 1

# 2PL probability
p = 1/(1 + np.exp(-a*(theta - b)))

# Likelihood
likelihood = p**y * (1-p)**(1-y)

# Bayesian update
posterior = prior * likelihood
posterior /= posterior.sum()

# Posterior statistics
posterior_mean = np.sum(theta * posterior)
posterior_map = theta[np.argmax(posterior)]

print(f"Posterior Mean = {posterior_mean:.3f}")
print(f"MAP Estimate   = {posterior_map:.3f}")

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta,
    y=prior,
    name="Prior"
))

fig.add_trace(go.Scatter(
    x=theta,
    y=posterior,
    name="Posterior"
))

fig.update_layout(
    title="Single Bayesian Update on a Fixed Grid",
    xaxis_title="Ability (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)

fig.show()

Posterior Mean = 0.675
MAP Estimate   = 0.661


##Task 7

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# -----------------------------
# Simulation Parameters
# -----------------------------

np.random.seed(42)

theta_true = 0.75
n = 20

# Ability grid
theta = np.linspace(-4, 4, 1000)

# Standard Normal Prior
posterior = norm.pdf(theta, 0, 1)
posterior /= posterior.sum()

# Store estimates
posterior_mean = []
posterior_map = []

# Initial estimates (Step 0)
posterior_mean.append(np.sum(theta * posterior))
posterior_map.append(theta[np.argmax(posterior)])

# -----------------------------
# Sequential Bayesian Updating
# -----------------------------

for k in range(n):

    # Random item parameters
    a = np.random.uniform(0.5, 2.0)
    b = np.random.normal(0, 1)

    # True probability of correct response
    p_true = 1 / (1 + np.exp(-a * (theta_true - b)))

    # Simulated user response
    y = np.random.rand() < p_true

    # Probability for every grid point
    p_grid = 1 / (1 + np.exp(-a * (theta - b)))

    # Likelihood
    likelihood = p_grid**y * (1-p_grid)**(1-y)

    # Bayesian update
    posterior *= likelihood
    posterior /= posterior.sum()

    # Posterior Mean
    mean = np.sum(theta * posterior)

    # MAP Estimate
    MAP = theta[np.argmax(posterior)]

    posterior_mean.append(mean)
    posterior_map.append(MAP)

# -----------------------------
# Plot
# -----------------------------

steps = np.arange(0, n+1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode='lines+markers',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map,
        mode='lines+markers',
        name='MAP Estimate'
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True Ability (0.75)"
)

fig.update_layout(
    title="Running Bayesian Ability Estimation",
    xaxis_title="Question Number",
    yaxis_title="Estimated Ability",
    template="plotly_white"
)

fig.show()

/tmp/ipykernel_2101/4143715604.py:49: DeprecationWarning:

In future, it will be an error for 'np.bool' scalars to be interpreted as an index



# Task 7: Evaluating Convergence over the Timeline

The simulation considers a user whose true latent ability is

$$
\theta_{\mathrm{true}}=0.75.
$$

Initially, the learning platform has no observations and therefore assumes the prior distribution

$$
\Theta\sim N(0,1).
$$

The posterior distribution is updated sequentially after every answered question.

For each item,

- the discrimination parameter is randomly generated from

$$
a_k\sim U(0.5,2.0),
$$

- the difficulty parameter is sampled from

$$
b_k\sim N(0,1),
$$

- the user's response is simulated according to the 2PL model

$$
P(Y_k=1|\theta_{\mathrm{true}})
=
\frac1{1+\exp[-a_k(\theta_{\mathrm{true}}-b_k)]}.
$$

A random number

$$
u\sim U(0,1)
$$

is generated.

If

$$
u<P(Y_k=1),
$$

the response is recorded as

$$
Y_k=1,
$$

otherwise

$$
Y_k=0.
$$

After every response, Bayesian updating is performed on the fixed ability grid.

The following two estimators are calculated:

### Posterior Mean

$$
\boxed{
\hat{\theta}_{\mathrm{Bayes}}
=
\sum_j
\theta_j
f(\theta_j|Y^{(k)})
}
$$

### Maximum A Posteriori Estimate

$$
\boxed{
\hat{\theta}_{\mathrm{MAP}}
=
\operatorname*{arg\,max}_{\theta_j}
f(\theta_j|Y^{(k)})
}
$$

Both estimators are recorded after each item and plotted over time together with the true latent ability.


# Analysis

At the beginning of the assessment, only the prior information is available.

Consequently, both the Posterior Mean and the MAP estimate are close to zero because the prior distribution is centered at

$$
\theta=0.
$$

As additional item responses are observed, more evidence about the user's latent ability becomes available.

Each Bayesian update modifies the posterior distribution according to both the correctness of the response and the characteristics of the administered item.

Items with high discrimination provide stronger evidence and therefore produce larger changes in the posterior distribution, whereas low-discrimination items contribute relatively little information.

Similarly, correctly answering difficult items shifts the posterior toward larger ability values more than correctly answering easy items.

As the number of observed responses increases, both the Posterior Mean and the MAP estimate gradually converge toward the true latent ability

$$
\theta_{\mathrm{true}}=0.75.
$$

Although small fluctuations may occur because responses are randomly simulated, the estimates generally become more stable over time.

This stabilization reflects the reduction in posterior uncertainty.

The posterior distribution becomes increasingly concentrated around the user's true ability, indicating that the learning platform becomes more confident in its measurement.

Therefore, the decreasing distance between the estimated ability and the true latent ability demonstrates successful Bayesian learning through sequential evidence accumulation.

# Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

##Answers

## 1. Structural Probability and Properties

The prior distribution of the unknown click-through rate

$$
\Theta \sim \mathrm{Beta}(\alpha,\beta)
$$

has probability density function

$$
f(\theta)
=
\frac{\Gamma(\alpha+\beta)}
{\Gamma(\alpha)\Gamma(\beta)}
\theta^{\alpha-1}(1-\theta)^{\beta-1},
\qquad
0\le\theta\le1.
$$

The Beta distribution is extremely flexible because its shape is entirely controlled by the parameters
$(\alpha)$ and $(\beta)$.

The mean of the distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

### Interpretation

**Case 1**

$$
(\alpha,\beta)=(1,1)
$$

produces the Uniform distribution.

There is no prior preference for any value of the CTR.

---

**Case 2**

$$
(\alpha,\beta)=(2,8)
$$

The density is concentrated near zero.

The prior believes that the advertisement is unlikely to receive clicks.

---

**Case 3**

$$
(\alpha,\beta)=(8,2)
$$

The density shifts toward one.

The prior assumes the advertisement performs well and has a high CTR.

As the ratio

$$
\frac{\alpha}{\beta}
$$

increases, the center of mass moves toward one.

Conversely, larger values of

$
\beta
$

relative to

$
\alpha
$

move the density toward zero.

In [1]:
import numpy as np
from scipy.stats import beta
import plotly.graph_objects as go

theta = np.linspace(0.001,0.999,500)

parameters = [
    (1,1,"Beta(1,1)"),
    (2,8,"Beta(2,8)"),
    (8,2,"Beta(8,2)")
]

fig = go.Figure()

for a,b,label in parameters:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode='lines',
            name=label
        )
    )

fig.update_layout(
    title="Beta Prior Distributions",
    xaxis_title="CTR (θ)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

## 2. Sequential Likelihood and Joint History

Each observation

$
Y_k
$

is Bernoulli distributed,

$$
Y_k\sim\mathrm{Bernoulli}(\theta).
$$

Therefore,

$$
P(Y_k=y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

Hence, the likelihood contribution of a single observation is

$$
L(y_k|\theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

Since observations are conditionally independent,

$$
L(\mathbf y^{(k)}|\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

Collecting exponents,

$$
L(\mathbf y^{(k)}|\theta)
=
\theta^{\sum y_i}
(1-\theta)^{k-\sum y_i}.
$$

## 3. Closed-Form Bayesian Update

Suppose that

$$
\Theta
\sim
\mathrm{Beta}
(\alpha_{k-1},\beta_{k-1}).
$$

Using Bayes' theorem,

$$
f(\theta|\mathbf y^{(k)})
\propto
L(y_k|\theta)
f(\theta|\mathbf y^{(k-1)}).
$$

Substituting,

$$
f(\theta|\mathbf y^{(k)})
\propto
\theta^{y_k}(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining exponents,

$$
f(\theta|\mathbf y^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

Hence,

$$
\boxed{
\Theta|\mathbf y^{(k)}
\sim
\mathrm{Beta}
(\alpha_k,\beta_k)
}
$$

where

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+1-y_k
}
$$

which proves Beta-Binomial conjugacy.

The posterior mean becomes

$$
\boxed{
E[\Theta|\mathbf y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}
$$

## 4. Dynamic Shifting Mechanics

If

$$
y_k=1,
$$

then

$$
\alpha_k=\alpha_{k-1}+1,
$$

while

$$
\beta_k
$$

remains unchanged.

Consequently,

$$
\frac{\alpha_k}
{\alpha_k+\beta_k}
$$

increases, causing the posterior density to shift toward one.

---

If

$$
y_k=0,
$$

then

$$
\beta_k=\beta_{k-1}+1,
$$

while

$$
\alpha_k
$$

remains unchanged.

The posterior therefore shifts toward zero.

---

Unlike this model, the 2PL Item Response Theory model has a nonlinear logistic likelihood,

$$
P(Y=1|\theta)
=
\frac{1}
{1+\exp[-a(\theta-b)]},
$$

which is not conjugate with any standard prior.

Therefore,

$$
f(\theta|y)
$$

cannot be obtained analytically and numerical methods such as grid approximation, MCMC, or Laplace approximation are required.

## 5. Running Point Estimators

### Posterior Mean

$$
\boxed{
\widehat\theta_{\text{Bayes}}^{(k)}
=
\frac{\alpha_k}
{\alpha_k+\beta_k}
}
$$

---

### Maximum A Posteriori (MAP)

When

$$
\alpha_k>1,\qquad
\beta_k>1,
$$

the MAP estimator is

$$
\boxed{
\widehat\theta_{\text{MAP}}^{(k)}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
}
$$

If either parameter is less than or equal to one, the posterior mode lies on the boundary of the interval.

## 6. Performance Tracking and Convergence Analysis

In [2]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.35
n = 100

alpha = 1
beta = 1

posterior_mean = [alpha/(alpha+beta)]
posterior_map = [0.5]

steps = [0]

for k in range(1,n+1):

    y = np.random.rand() < theta_true

    alpha += int(y)
    beta += 1-int(y)

    mean = alpha/(alpha+beta)

    if alpha>1 and beta>1:
        MAP = (alpha-1)/(alpha+beta-2)
    else:
        MAP = mean

    posterior_mean.append(mean)
    posterior_map.append(MAP)
    steps.append(k)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode='lines',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map,
        mode='lines',
        name='MAP'
    )
)

fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True CTR'
)

fig.update_layout(
    title='Sequential Bayesian Estimation of CTR',
    xaxis_title='Impression',
    yaxis_title='Estimated CTR',
    template='plotly_white'
)

fig.show()

## Analysis

Initially, the prior

$$
\mathrm{Beta}(1,1)
$$

represents complete uncertainty.

Each observed click increases

$$
\alpha,
$$

whereas each non-click increases

$$
\beta.
$$

Consequently, the posterior mean and MAP estimator gradually move toward the true click-through rate.

As

$$
k\rightarrow100,
$$

the influence of the prior becomes negligible compared with the accumulated observations.

Both estimators converge toward the true parameter

$$
\theta_{\text{true}}=0.35,
$$

illustrating Bayesian learning through evidence accumulation.

This demonstrates that, under a conjugate Beta-Binomial model, sequential Bayesian estimation can be performed analytically without numerical integration, making it computationally efficient for real-time CTR tracking.